<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Iterative Updates for the Clock Settings

**Reduce squared clock error by repeatedly moving against the slope.**

Keep the four clock observations, corrected-error calculation, and squared-error objective. Remove the calibration equality and correction bound; the settings are unrestricted unless the rate is explicitly held at zero. Hold the rate at zero to isolate one correction update. An iteration counts a computation, not elapsed physical time.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

This is a smooth, unconstrained least-squares problem and a quadratic program (QP). These names describe the problem form; gradient descent and Newton's method are solution procedures.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.

With $r=0$ held fixed, the one-setting objective is

$$
f(q,0)=(q-2)^2+2(q-1)^2+q^2=4(q-1)^2+2.
$$

This identity gives the exact reference $q^\star=1$ min and $f(q^\star,0)=2\ \mathrm{min}^2$. Every numerical candidate can be compared with this exact reference. The observed data and response calculation remain fixed as the candidate changes.


In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Use the slope to choose an update

An **iterate** is a candidate produced by the algorithm. Write $q^{(k)}$ for the candidate after $k$ updates, starting from $q^{(0)}=0$. The superscript counts computations; it is not elapsed clock time or a power. These candidates are proposed settings, not physical states carried forward by the clock.

The derivative of the offset-only objective is

$$
\frac{\partial f}{\partial q}(q,0)=8(q-1).
$$

A negative derivative says that a small increase in $q$ lowers the score. A positive derivative says that a small decrease lowers it. **Gradient descent** moves against that slope:

$$
q^{(k+1)}=q^{(k)}-\alpha\,8(q^{(k)}-1).
$$

The positive step size $\alpha$ is an analyst-chosen **hyperparameter**. It changes the search trajectory, not the corrected errors of a fixed decision.

With $\alpha=0.05$, the slope at zero is $-8$, so the first update is $q^{(1)}=0-0.05(-8)=0.4$. The score falls from 6 to $4(0.4-1)^2+2=3.44$. The next update gives $q^{(2)}=0.4-0.05(-4.8)=0.64$, with score 2.5184. The arrows below show the change in the setting and its lower score on the same curve.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/offset_descent.svg" alt="On the offset-only squared-error curve, the initial correction 0 has score 6. A step against the negative slope moves right to correction 0.4 with score 3.44. The exact minimum is at correction 1 with score 2." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The recurrence below stores every proposed correction, starting at zero. The observations and rate remain fixed.


In [ ]:
def offset_history(step_size, steps, initial_correction=0.0):
    correction = float(initial_correction)
    history = [correction]
    for _ in range(steps):
        slope = 8.0 * (correction - 1.0)
        correction -= step_size * slope
        history.append(correction)
    return np.array(history)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The calculation reproduces the first two updates without changing the observations or rate setting.

In [ ]:
for iteration, correction in enumerate(offset_history(0.05, 2)):
    result = evaluate_clock((correction, 0.0))
    print(f"k={iteration}: q={correction:.4f} min, "
          f"f={result['objective']:.4f} min², offset slope={8*(correction-1):.4f}")

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit. This notebook develops the first two offset-only updates using the established clock example.

The source example's clipping operation is omitted because these settings are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.
